# Preliminary: Analysis for the cosmology benchmark (C functions)

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

In [ ]:
x1, x2 = sympy.symbols("x1:3")
x = sympy.abc.x
y = sympy.abc.y

## Loading data

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report.run_set.unique()

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
f_indices = full_report.data_set.apply(lambda x: x.startswith("C"))
fr2 = full_report.loc[f_indices].set_index(["run_set", "data_set", "sample_num"])

In [ ]:
fr2["sympy"] = fr2.expr_original_syms.apply(au.parse_if_needed)
fr2["sympy_defuzz"] = fr2.expr_original_syms_defuzz.apply(au.parse_if_needed)

Out of all the run sets, these are the two that will be analyzed and reported.

In [ ]:
srb_key = "SRB-2026-07-20-1300"
cht_key = "CHT-2026-07-20-1300"

In [ ]:
srb = fr2.loc[srb_key]
cht = fr2.loc[cht_key]

In [ ]:
srb.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
srb_min_mse_ixs = srb.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb.loc[srb_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht_min_mse_ixs = cht.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb.groupby(by="data_set").agg(
    complexity_mean=("complexity", "mean"),
    complexity_defuzz_mean=("complexity_defuzz", "mean"),
    complexity_std=("complexity", "std"),
    complexity_defuzz_std=("complexity_defuzz", "std"),
    complexity_min=("complexity", "min"),
    complexity_max=("complexity", "max"),
    complexity_defuzz_min=("complexity_defuzz", "min"),
    complexity_defuzz_max=("complexity_defuzz", "max"),
).sort_values("complexity_min", ascending=True)

In [ ]:
srb_threshold_table = au.mse_threshold_table(srb)
srb_threshold_table

In [ ]:
cht_threshold_table = au.mse_threshold_table(cht)
cht_threshold_table

In [ ]:
au.to_latex(
    srb_threshold_table,
    file="Generated/srb_threshold_table.tex",
    strip_colname_prefix="mse",
)

In [ ]:
au.to_latex(
    cht_threshold_table,
    file="Generated/cht_threshold_table.tex",
    strip_colname_prefix="mse",
)

## Polynomials

In [ ]:
data_sets_polynomial = ["C2a"]

In [ ]:
srb.loc[data_sets_polynomial]

`C2a` is no problem.

I used to include `C5f` here, but it's a rational function, not a polynomial.
Table on p31 of the cosmology article is confusing, because `C2a` has a reference to $H(z)$ but there's a column of $H$ in the data file and it really is a simple polynomial.
But `C5f`, which looks like the same kind of item, seems to be referring to `C5d` and `C5e` as functions or $R_0$ and $r$.

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
(srb.loc[data_sets_polynomial]
 .sympy_defuzz.apply(lambda e: not e.is_polynomial(x1, x2))
 .groupby(level="data_set")
 .sum())

All runs on all polynomial data sets are correct up to fuzz.

## Rational functions

In [ ]:
data_sets_rational = ["C3g", "C3h", "C5a", "C5b", "C5c", "C5d", "C5e", "C5f"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C3g: no, but MSE is better than reference
C3h: no, but MSE is better than reference
C5a: perfect
C5b: no
C5c: no
C5d: perfect but in a different form, see below
C5e: no, but MSE is not bad
C5f: no
```

In [ ]:
C5d_best = srb.loc[srb_min_mse_ixs].loc["C5d"].sympy_defuzz.iloc[0]

In [ ]:
C5d_best

In [ ]:
sympy.simplify(sympy.together(C5d_best))

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C3g: no, but MSE is better than reference
C3h: perfect after defuzzing
C5a: perfect
C5b: no
C5c: no
C5d: perfect
C5e: no, but MSE is not bad
C5f: 
```

In [ ]:
C3h_cht_best = cht.loc[cht_min_mse_ixs].loc["C3h"].sympy_defuzz.iloc[0]

In [ ]:
au.replace_near_integer(sympy.expand(C3h_cht_best))

In [ ]:
C5b_cht_best = cht.loc[cht_min_mse_ixs].loc["C5b"].sympy_defuzz.iloc[0]

In [ ]:
sympy.together(C5b_cht_best)

In [ ]:
C5d_cht_best = cht.loc[cht_min_mse_ixs].loc["C5d"].sympy_defuzz.iloc[0]

In [ ]:
sympy.simplify(sympy.together(sympy.expand(C5d_cht_best)))

In [ ]:
C5e_cht_best = cht.loc[cht_min_mse_ixs].loc["C5e"].sympy_defuzz.iloc[0]

In [ ]:
sympy.expand(C5e_cht_best)

The majority of solutions are not rational functions.

In [ ]:
(srb.loc[data_sets_rational]
 .sympy_defuzz
 .apply(lambda e: not e.is_rational_function())
 .groupby(level="data_set").sum())

In [ ]:
rational_plot_params = {
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 399),
    "complexity_binwidth": 20,
    "mse_lims": (1.0e-32, 1.e2),
    "mse_binwidth": 2.0,
    "xlabel": "Complexity (defuzz)",
    "ylabel": "MSE"
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_rational],
    file_stem="srb-rational-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_rational],
    file_stem="cht-rational-complexity-mse-displot",
    **rational_plot_params
)

## Power functions

In [ ]:
data_sets_power = ["C1a", "C1b", "C1c", "C1d", "C2b"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_power, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C1a: no
C1b: no, although it has a term of roughly the correct form.
C1c: no
C1d: no
C2b: no
````

In [ ]:
C1b_best = srb.loc[srb_min_mse_ixs].loc["C1b"].sympy_defuzz.iloc[0]

In [ ]:
C1b_best

In [ ]:
C1c_best = srb.loc[srb_min_mse_ixs].loc["C1c"].sympy_defuzz.iloc[0]

In [ ]:
C1c_best

In [ ]:
au.replace_near_integer(C1c_best, tolerance=1.0e-4)

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_power, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Kind of the same.

Let's take a look at the C1a data.
It's just a slight curve.

In [ ]:
C1a_df = pd.read_csv("Things-to-bench/cosmo_data/C1a.csv")

In [ ]:
sns.scatterplot(data=C1a_df, x="z", y="target")

I bet if we gave it something with more shape it would have better luck.

In [ ]:
xs = np.linspace(-2.2, 2, 100)
ys = 0.0718 * np.sqrt(0.3 * (1 + xs) ** 3 + 0.7)

In [ ]:
sns.scatterplot(x=xs, y=ys)

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_power],
    file_stem="srb-power-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_power],
    file_stem="cht-power-complexity-mse-displot",
    **rational_plot_params
)

## Hard

In [ ]:
data_sets_hard = ["C6a", "C6b", "C6c"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_hard, ["mse", "complexity_defuzz", "sympy_defuzz"]]

```
C6a: nope
C6b: decent MSE, better than cp3 article, some promising terms
C6c: nope
```

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_hard, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Nope.

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_hard],
    file_stem="srb-hard-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_hard],
    file_stem="cht-hard-complexity-mse-displot",
    **rational_plot_params
)

## Black box

In [ ]:
data_sets_black_box = ["C3a", "C3b", "C3c", "C3d", "C3e", "C3f", "C4a", "C4b", "C4c", "C4d", "C4e"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_black_box, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In the cp3 article, there's no given exact form for these problems, just an order of magnitude for MSE for the best result.
So the best I can do is compare MSE by the power of 10.
```
C3a: better than cp3
C3b: better than cp3
C3c: similar to cp3
C3d: better than cp3
C3e: similar to cp3
C3f: better than cp3
C4a: better than cp3
C4b: similar to cp3
C4c: similar to cp3
C4d: similar to cp3
C4e: worse than cp3
```

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_black_box, ["mse", "complexity_defuzz", "sympy_defuzz"]]

Cheating isn't really possible because we don't know what hint to give.

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_black_box],
    file_stem="srb-black-box-complexity-mse-displot",
    **rational_plot_params
)